In [ ]:

"""
CN7 12 - Supervised 6,736행 Provenance / Test Coverage 진단
===========================================================

왜 하는가
---------
현재 moldset_labeled_cn7:
    1,211 rows / 17 defects

하지만 supervised_label_cn7:
    약 6,736 rows / 39 defects

가 별도로 존재한다.

CN7 test는 복원 결과:
    35,239 rows
    2020-03 ~ 2020-10
    EQUIP_CD = S14
    650톤-우진2호기

즉 1,211행보다 supervised 6,736행이 test 생산영역을
더 넓게 덮는지 확인해야 한다.

이번 단계
---------
1) labeled_data.csv에서 CN7 source subset 재구성
2) supervised_label_cn7.csv와 exact sensor multiset 비교
3) source target 의미 복원
4) 날짜 / Serial / Side / Equipment coverage 확인
5) test raw 35,239와 sensor distribution shift 비교
6) supervised ↔ test exact sensor-vector overlap 확인
7) defect vector가 test에 얼마나 등장하는지 확인

모델 학습 없음.
"""

from pathlib import Path
from collections import Counter
import warnings

import numpy as np
import pandas as pd

from scipy.stats import ks_2samp, wasserstein_distance

warnings.filterwarnings("ignore")


# =============================================================================
# 0. 경로
# =============================================================================

BASE_DIR = Path(
    r"../../../data/raw"
)

CN7_WORK_DIR = Path(
    r"../../outputs/research"
)

LABELED_SOURCE_PATH = (
    BASE_DIR
    / "labeled_data.csv"
)

SUPERVISED_PATH = (
    BASE_DIR
    / "supervised_label_cn7.csv"
)

TEST_RESTORED_PATH = (
    CN7_WORK_DIR
    / "cn7_10_test_restore"
    / "cn7_test_restored_35239.csv"
)

MOLDSET_TRAIN_PATH = (
    CN7_WORK_DIR
    / "cn7_03_eda"
    / "cn7_train_raw_1211.csv"
)

OUT_DIR = (
    CN7_WORK_DIR
    / "cn7_12_supervised_coverage"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TARGET = "PassOrFail"

SENSOR_COLS = [
    "Injection_Time",
    "Filling_Time",
    "Plasticizing_Time",
    "Cycle_Time",
    "Clamp_Close_Time",
    "Cushion_Position",
    "Plasticizing_Position",
    "Clamp_Open_Position",
    "Max_Injection_Speed",
    "Max_Screw_RPM",
    "Average_Screw_RPM",
    "Max_Injection_Pressure",
    "Max_Switch_Over_Pressure",
    "Max_Back_Pressure",
    "Average_Back_Pressure",
    "Barrel_Temperature_1",
    "Barrel_Temperature_2",
    "Barrel_Temperature_3",
    "Barrel_Temperature_4",
    "Barrel_Temperature_5",
    "Barrel_Temperature_6",
    "Hopper_Temperature",
    "Mold_Temperature_3",
    "Mold_Temperature_4",
]


# =============================================================================
# 유틸
# =============================================================================

def banner(title):
    print(
        "\n"
        + "=" * 118
    )
    print(title)
    print(
        "=" * 118
    )


def extract_date(series):
    """
    '2020-10-16 오전 12:00:00'처럼
    pandas가 바로 못 읽는 문자열도 날짜 부분만 안정적으로 추출.
    """
    text = (
        series
        .astype(str)
    )

    date_text = (
        text
        .str.extract(
            r"(\d{4}-\d{2}-\d{2})",
            expand=False,
        )
    )

    return pd.to_datetime(
        date_text,
        errors="coerce",
    )


def infer_side(df):
    if "Side" in df.columns:
        return (
            df["Side"]
            .astype(str)
        )

    if "PART_NAME" in df.columns:
        s = (
            df["PART_NAME"]
            .astype(str)
            .str.upper()
        )

        return pd.Series(
            np.where(
                s.str.contains(
                    r"\bRH\b",
                    regex=True,
                    na=False,
                ),
                "RH",
                np.where(
                    s.str.contains(
                        r"\bLH\b",
                        regex=True,
                        na=False,
                    ),
                    "LH",
                    "UNKNOWN",
                ),
            ),
            index=df.index,
        )

    return pd.Series(
        "UNKNOWN",
        index=df.index,
    )


def normalize_source_target(df):
    """
    source PassOrFail의 defect label을 Reason과 함께 데이터에서 추론.

    Reason이 존재하는 label을 defect로 간주한다.
    """
    if TARGET not in df.columns:
        raise KeyError(
            f"{TARGET} 없음"
        )

    raw = df[TARGET]

    if "Reason" in df.columns:
        reason_present = (
            df["Reason"]
            .notna()
            &
            df["Reason"]
            .astype(str)
            .str.strip()
            .ne("")
        )

        evidence = (
            pd.DataFrame({
                "label":
                    raw.astype(str),
                "reason_present":
                    reason_present.astype(int),
            })
            .groupby(
                "label"
            )
            .agg(
                rows=(
                    "reason_present",
                    "size",
                ),
                reason_rows=(
                    "reason_present",
                    "sum",
                ),
                reason_rate=(
                    "reason_present",
                    "mean",
                ),
            )
            .sort_values(
                "reason_rate",
                ascending=False,
            )
        )

        print(
            "Source target / Reason evidence:"
        )

        print(
            evidence.to_string()
        )

        defect_label = (
            evidence.index[0]
        )

        print(
            "Reason 기준 defect label:",
            defect_label,
        )

        normalized = (
            raw.astype(str)
            == str(defect_label)
        ).astype(int)

        return normalized, defect_label

    # Reason 없는 경우 fallback
    values = (
        raw.dropna()
        .unique()
        .tolist()
    )

    values_str = {
        str(v).upper():
            v
        for v in values
    }

    if (
        "N" in values_str
        and "Y" in values_str
    ):
        return (
            raw.astype(str)
            .str.upper()
            .eq("N")
            .astype(int),
            "N",
        )

    numeric = pd.to_numeric(
        raw,
        errors="coerce",
    )

    if set(
        numeric.dropna().unique()
    ).issubset({
        0,
        1,
    }):
        return (
            numeric.astype(int),
            1,
        )

    raise RuntimeError(
        "source target 의미를 자동 추론할 수 없습니다."
    )


def rounded_keys(
    df,
    cols,
    decimals=8,
):
    arr = (
        df[
            cols
        ]
        .astype(float)
        .to_numpy()
    )

    arr = np.round(
        arr,
        decimals,
    )

    return [
        tuple(row)
        for row in arr
    ]


def psi_numeric(
    train,
    test,
    bins=10,
):
    train = np.asarray(
        train,
        dtype=float,
    )

    test = np.asarray(
        test,
        dtype=float,
    )

    train = train[
        np.isfinite(train)
    ]

    test = test[
        np.isfinite(test)
    ]

    if (
        len(train) == 0
        or len(test) == 0
    ):
        return np.nan

    qs = np.linspace(
        0,
        1,
        bins + 1,
    )

    edges = np.unique(
        np.quantile(
            train,
            qs,
        )
    )

    if len(edges) < 3:
        return 0.0

    edges[0] = -np.inf
    edges[-1] = np.inf

    a, _ = np.histogram(
        train,
        bins=edges,
    )

    b, _ = np.histogram(
        test,
        bins=edges,
    )

    a = a / a.sum()
    b = b / b.sum()

    eps = 1e-6

    a = np.clip(
        a,
        eps,
        None,
    )

    b = np.clip(
        b,
        eps,
        None,
    )

    return float(
        np.sum(
            (b - a)
            * np.log(
                b / a
            )
        )
    )


def multiset_equal(
    a_df,
    b_df,
    cols,
):
    if len(a_df) != len(b_df):
        return (
            False,
            None,
        )

    for decimals in [
        10,
        9,
        8,
        7,
        6,
    ]:
        a_keys = Counter(
            rounded_keys(
                a_df,
                cols,
                decimals,
            )
        )

        b_keys = Counter(
            rounded_keys(
                b_df,
                cols,
                decimals,
            )
        )

        if a_keys == b_keys:
            return (
                True,
                decimals,
            )

    return (
        False,
        None,
    )


# =============================================================================
# 1. 로드
# =============================================================================

banner(
    "CN7 12 - Supervised 6,736행 Coverage 진단"
)

source = pd.read_csv(
    LABELED_SOURCE_PATH
)

supervised = pd.read_csv(
    SUPERVISED_PATH
)

test = pd.read_csv(
    TEST_RESTORED_PATH
)

moldset_train = pd.read_csv(
    MOLDSET_TRAIN_PATH
)

print(
    "labeled_data:",
    source.shape,
)

print(
    "supervised_label_cn7:",
    supervised.shape,
)

print(
    "test restored:",
    test.shape,
)

print(
    "moldset train:",
    moldset_train.shape,
)


# =============================================================================
# 2. labeled_data -> CN7
# =============================================================================

banner(
    "1) labeled_data CN7 subset"
)

if "PART_NAME" not in source.columns:
    raise KeyError(
        "labeled_data에 PART_NAME 없음"
    )

cn7_mask = (
    source[
        "PART_NAME"
    ]
    .astype(str)
    .str.contains(
        "CN7",
        case=False,
        na=False,
    )
)

source_cn7 = (
    source.loc[
        cn7_mask
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

print(
    "source CN7:",
    source_cn7.shape,
)

print(
    "supervised:",
    supervised.shape,
)

print(
    "row count same:",
    len(source_cn7)
    == len(supervised),
)


# =============================================================================
# 3. 공통 센서 exact multiset 검증
# =============================================================================

banner(
    "2) Source CN7 ↔ supervised exact sensor multiset"
)

sensor_cols = [
    c
    for c in SENSOR_COLS
    if (
        c in source_cn7.columns
        and c in supervised.columns
        and c in test.columns
    )
]

print(
    "common sensors:",
    len(sensor_cols),
)

match, decimals = multiset_equal(
    source_cn7,
    supervised,
    sensor_cols,
)

print(
    "sensor multiset exact:",
    match,
)

print(
    "matched decimals:",
    decimals,
)


# =============================================================================
# 4. Target 의미 복원
# =============================================================================

banner(
    "3) Target 의미 / 불량 수"
)

source_target, source_defect_label = (
    normalize_source_target(
        source_cn7
    )
)

source_cn7[
    "__target_norm__"
] = source_target

supervised_target = pd.to_numeric(
    supervised[TARGET],
    errors="raise",
).astype(int)

print()

print(
    "source normalized target:"
)

print(
    source_cn7[
        "__target_norm__"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)

print()

print(
    "supervised target:"
)

print(
    supervised_target
    .value_counts()
    .sort_index()
    .to_string()
)

print()

print(
    "target count same:",
    source_cn7[
        "__target_norm__"
    ].value_counts(
        dropna=False
    ).sort_index().to_dict()
    ==
    supervised_target.value_counts(
        dropna=False
    ).sort_index().to_dict()
)

# sensor + target multiset까지 검증
source_for_target = (
    source_cn7[
        sensor_cols
    ]
    .copy()
)

source_for_target[
    "__target__"
] = source_cn7[
    "__target_norm__"
].to_numpy()

supervised_for_target = (
    supervised[
        sensor_cols
    ]
    .copy()
)

supervised_for_target[
    "__target__"
] = supervised_target.to_numpy()

target_match, target_decimals = (
    multiset_equal(
        source_for_target,
        supervised_for_target,
        sensor_cols
        + ["__target__"],
    )
)

print(
    "sensor + target multiset exact:",
    target_match,
)

print(
    "sensor + target decimals:",
    target_decimals,
)


# =============================================================================
# 5. Metadata coverage
# =============================================================================

banner(
    "4) Supervised source metadata coverage"
)

source_cn7[
    "__Side__"
] = infer_side(
    source_cn7
)

test[
    "__Side__"
] = infer_side(
    test
)

# TimeStamp
for name, d in [
    (
        "SUPERVISED_SOURCE",
        source_cn7,
    ),
    (
        "MOLDSET_1211",
        moldset_train,
    ),
    (
        "TEST_35239",
        test,
    ),
]:

    print(
        f"\n[{name}]"
    )

    if "TimeStamp" in d.columns:

        ts = pd.to_datetime(
            d[
                "TimeStamp"
            ],
            errors="coerce",
        )

        print(
            "TimeStamp:",
            ts.min(),
            "~",
            ts.max(),
        )

        print(
            "unique Timestamp dates:",
            ts.dt.date.nunique(),
        )

    if (
        "PART_FACT_PLAN_DATE"
        in d.columns
    ):

        plan = extract_date(
            d[
                "PART_FACT_PLAN_DATE"
            ]
        )

        print(
            "PlanDate:",
            plan.min(),
            "~",
            plan.max(),
        )

        print(
            "unique PlanDate:",
            plan.dt.date.nunique(),
        )

    if (
        "PART_FACT_SERIAL"
        in d.columns
    ):

        print(
            "Serial unique:",
            d[
                "PART_FACT_SERIAL"
            ].nunique(
                dropna=False
            ),
        )

    side = infer_side(
        d
    )

    print(
        "Side:"
    )

    print(
        side
        .value_counts()
        .to_string()
    )

    if "EQUIP_CD" in d.columns:

        print(
            "Equipment:"
        )

        print(
            d[
                "EQUIP_CD"
            ]
            .value_counts(
                dropna=False
            )
            .head(20)
            .to_string()
        )


# =============================================================================
# 6. Defect distribution by metadata
# =============================================================================

banner(
    "5) Supervised source defect coverage"
)

defect_meta_tables = {}

for c in [
    "__Side__",
    "PART_FACT_SERIAL",
    "EQUIP_CD",
    "EQUIP_NAME",
]:
    if c in source_cn7.columns:

        tab = (
            pd.crosstab(
                source_cn7[c],
                source_cn7[
                    "__target_norm__"
                ],
                margins=False,
            )
        )

        defect_meta_tables[
            c
        ] = tab

        print(
            f"\n[{c}]"
        )

        print(
            tab.to_string()
        )

# 날짜별
if "TimeStamp" in source_cn7.columns:

    source_cn7[
        "__date__"
    ] = pd.to_datetime(
        source_cn7[
            "TimeStamp"
        ],
        errors="coerce",
    ).dt.date

    date_tab = pd.crosstab(
        source_cn7[
            "__date__"
        ],
        source_cn7[
            "__target_norm__"
        ],
    )

    print(
        "\n[TimeStamp Date]"
    )

    print(
        date_tab.to_string()
    )

    date_tab.to_csv(
        OUT_DIR
        / "supervised_defect_by_date.csv",
        encoding="utf-8-sig",
    )


# =============================================================================
# 7. Metadata overlap train ↔ test
# =============================================================================

banner(
    "6) Metadata coverage vs Test"
)

coverage_rows = []

# Serial
if (
    "PART_FACT_SERIAL"
    in source_cn7.columns
    and "PART_FACT_SERIAL"
    in test.columns
):

    tr_serial = set(
        source_cn7[
            "PART_FACT_SERIAL"
        ]
        .dropna()
        .tolist()
    )

    te_serial = set(
        test[
            "PART_FACT_SERIAL"
        ]
        .dropna()
        .tolist()
    )

    inter = (
        tr_serial
        & te_serial
    )

    print(
        "Serial:"
    )

    print(
        " supervised:",
        sorted(
            tr_serial,
            key=str,
        )
    )

    print(
        " test      :",
        sorted(
            te_serial,
            key=str,
        )
    )

    print(
        " intersection:",
        sorted(
            inter,
            key=str,
        )
    )

    print(
        " test serial coverage:",
        len(inter),
        "/",
        len(te_serial),
    )

    coverage_rows.append({
        "dimension":
            "PART_FACT_SERIAL",
        "train_unique":
            len(tr_serial),
        "test_unique":
            len(te_serial),
        "intersection":
            len(inter),
        "test_category_coverage":
            (
                len(inter)
                / len(te_serial)
                if te_serial
                else np.nan
            ),
    })


# PlanDate
if (
    "PART_FACT_PLAN_DATE"
    in source_cn7.columns
    and "PART_FACT_PLAN_DATE"
    in test.columns
):

    tr_date = set(
        extract_date(
            source_cn7[
                "PART_FACT_PLAN_DATE"
            ]
        )
        .dropna()
        .dt.date
        .tolist()
    )

    te_date = set(
        extract_date(
            test[
                "PART_FACT_PLAN_DATE"
            ]
        )
        .dropna()
        .dt.date
        .tolist()
    )

    inter = (
        tr_date
        & te_date
    )

    print()

    print(
        "PlanDate:"
    )

    print(
        " supervised unique:",
        len(tr_date),
    )

    print(
        " test unique:",
        len(te_date),
    )

    print(
        " intersection:",
        len(inter),
    )

    print(
        " test date coverage:",
        (
            len(inter)
            / len(te_date)
            if te_date
            else np.nan
        ),
    )

    coverage_rows.append({
        "dimension":
            "PART_FACT_PLAN_DATE",
        "train_unique":
            len(tr_date),
        "test_unique":
            len(te_date),
        "intersection":
            len(inter),
        "test_category_coverage":
            (
                len(inter)
                / len(te_date)
                if te_date
                else np.nan
            ),
    })


coverage_df = pd.DataFrame(
    coverage_rows
)

coverage_df.to_csv(
    OUT_DIR
    / "metadata_test_coverage.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 8. Sensor shift: supervised 6736 vs test
# =============================================================================

banner(
    "7) Supervised 6736 vs Test sensor shift"
)

shift_rows = []

for c in sensor_cols:

    tr = pd.to_numeric(
        source_cn7[c],
        errors="coerce",
    ).dropna()

    te = pd.to_numeric(
        test[c],
        errors="coerce",
    ).dropna()

    tr_std = float(
        tr.std(
            ddof=0
        )
    )

    if tr_std > 1e-12:
        smd = float(
            (
                te.mean()
                - tr.mean()
            )
            / tr_std
        )

        norm_w = float(
            wasserstein_distance(
                tr,
                te,
            )
            / tr_std
        )

    else:
        smd = np.nan
        norm_w = np.nan

    ks = ks_2samp(
        tr,
        te,
    )

    outside = float(
        (
            (te < tr.min())
            |
            (te > tr.max())
        ).mean()
    )

    shift_rows.append({
        "feature":
            c,
        "SMD_test_minus_supervised":
            smd,
        "KS_stat":
            float(
                ks.statistic
            ),
        "KS_pvalue":
            float(
                ks.pvalue
            ),
        "Norm_Wasserstein":
            norm_w,
        "PSI":
            psi_numeric(
                tr,
                te,
                bins=10,
            ),
        "Test_Outside_Supervised_Range":
            outside,
    })

shift_df = pd.DataFrame(
    shift_rows
)

shift_df[
    "abs_SMD"
] = shift_df[
    "SMD_test_minus_supervised"
].abs()

shift_df = (
    shift_df
    .sort_values(
        [
            "KS_stat",
            "abs_SMD",
        ],
        ascending=False,
    )
)

print(
    shift_df[
        [
            "feature",
            "SMD_test_minus_supervised",
            "KS_stat",
            "Norm_Wasserstein",
            "PSI",
            "Test_Outside_Supervised_Range",
        ]
    ]
    .round(4)
    .to_string(
        index=False
    )
)

shift_df.to_csv(
    OUT_DIR
    / "supervised_vs_test_shift.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 9. Exact sensor-vector overlap
# =============================================================================

banner(
    "8) Supervised ↔ Test exact sensor-vector overlap"
)

# train에서 상수인 feature는 exact key에서 제외
key_cols = [
    c
    for c in sensor_cols
    if source_cn7[
        c
    ].nunique(
        dropna=False
    ) > 1
]

train_keys = rounded_keys(
    source_cn7,
    key_cols,
    decimals=8,
)

test_keys = rounded_keys(
    test,
    key_cols,
    decimals=8,
)

train_counter = Counter(
    train_keys
)

test_counter = Counter(
    test_keys
)

train_unique = set(
    train_counter
)

test_unique = set(
    test_counter
)

common = (
    train_unique
    & test_unique
)

test_overlap_rows = int(
    sum(
        count
        for key, count
        in test_counter.items()
        if key in train_unique
    )
)

print(
    "supervised unique vectors:",
    len(
        train_unique
    ),
)

print(
    "test unique vectors:",
    len(
        test_unique
    ),
)

print(
    "common unique vectors:",
    len(
        common
    ),
)

print(
    "test exact-overlap rows:",
    test_overlap_rows,
    "/",
    len(test),
    "=",
    test_overlap_rows
    / len(test),
)


# =============================================================================
# 10. Target category별 vector overlap
# =============================================================================

banner(
    "9) Defect / Normal / Conflict vector overlap"
)

tmp = source_cn7.copy()

tmp[
    "__sensor_key__"
] = train_keys

vector_target = (
    tmp.groupby(
        "__sensor_key__"
    )[
        "__target_norm__"
    ]
    .agg(
        [
            "count",
            "sum",
            "nunique",
        ]
    )
)

positive_only_keys = set(
    vector_target.index[
        (
            vector_target[
                "sum"
            ] > 0
        )
        &
        (
            vector_target[
                "nunique"
            ] == 1
        )
    ]
)

negative_only_keys = set(
    vector_target.index[
        (
            vector_target[
                "sum"
            ] == 0
        )
        &
        (
            vector_target[
                "nunique"
            ] == 1
        )
    ]
)

conflict_keys = set(
    vector_target.index[
        vector_target[
            "nunique"
        ] > 1
    ]
)

def test_rows_matching(
    keyset
):
    return int(
        sum(
            test_counter[
                key
            ]
            for key in keyset
            if key in test_counter
        )
    )


positive_overlap = test_rows_matching(
    positive_only_keys
)

negative_overlap = test_rows_matching(
    negative_only_keys
)

conflict_overlap = test_rows_matching(
    conflict_keys
)

print(
    "positive-only vectors:",
    len(
        positive_only_keys
    ),
)

print(
    "negative-only vectors:",
    len(
        negative_only_keys
    ),
)

print(
    "same-X/different-target vectors:",
    len(
        conflict_keys
    ),
)

print()

print(
    "TEST rows matching positive-only:",
    positive_overlap,
)

print(
    "TEST rows matching negative-only:",
    negative_overlap,
)

print(
    "TEST rows matching conflict:",
    conflict_overlap,
)

pd.DataFrame([
    {
        "category":
            "positive_only",
        "train_unique_vectors":
            len(
                positive_only_keys
            ),
        "test_overlap_rows":
            positive_overlap,
    },
    {
        "category":
            "negative_only",
        "train_unique_vectors":
            len(
                negative_only_keys
            ),
        "test_overlap_rows":
            negative_overlap,
    },
    {
        "category":
            "same_X_different_target",
        "train_unique_vectors":
            len(
                conflict_keys
            ),
        "test_overlap_rows":
            conflict_overlap,
    },
]).to_csv(
    OUT_DIR
    / "test_overlap_by_supervised_target.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 11. 1211 vs 6736 coverage 비교
# =============================================================================

banner(
    "10) 1211 vs 6736 coverage 비교"
)

comparison_rows = []

for name, d in [
    (
        "moldset_1211",
        moldset_train,
    ),
    (
        "supervised_6736",
        source_cn7,
    ),
]:

    row = {
        "dataset":
            name,
        "rows":
            len(d),
    }

    if TARGET in d.columns:

        if name == "supervised_6736":

            row[
                "defects"
            ] = int(
                source_cn7[
                    "__target_norm__"
                ].sum()
            )

        else:

            row[
                "defects"
            ] = int(
                pd.to_numeric(
                    d[TARGET],
                    errors="coerce",
                )
                .fillna(0)
                .sum()
            )

    if "TimeStamp" in d.columns:

        ts = pd.to_datetime(
            d[
                "TimeStamp"
            ],
            errors="coerce",
        )

        row[
            "timestamp_min"
        ] = ts.min()

        row[
            "timestamp_max"
        ] = ts.max()

        row[
            "timestamp_unique_dates"
        ] = ts.dt.date.nunique()

    if (
        "PART_FACT_SERIAL"
        in d.columns
    ):

        row[
            "serial_unique"
        ] = d[
            "PART_FACT_SERIAL"
        ].nunique(
            dropna=False
        )

    comparison_rows.append(
        row
    )

comparison_df = pd.DataFrame(
    comparison_rows
)

print(
    comparison_df
    .to_string(
        index=False
    )
)

comparison_df.to_csv(
    OUT_DIR
    / "moldset1211_vs_supervised6736.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 12. 자동 요약
# =============================================================================

banner(
    "11) 자동 요약"
)

high_shift = shift_df[
    (
        shift_df[
            "KS_stat"
        ] >= 0.30
    )
    |
    (
        shift_df[
            "PSI"
        ] >= 0.25
    )
]

lines = []

lines.append(
    f"- source CN7 rows: "
    f"{len(source_cn7)}"
)

lines.append(
    f"- supervised rows: "
    f"{len(supervised)}"
)

lines.append(
    f"- sensor multiset exact: "
    f"{match}"
)

lines.append(
    f"- sensor+target multiset exact: "
    f"{target_match}"
)

lines.append(
    f"- supervised defects: "
    f"{int(source_cn7['__target_norm__'].sum())}"
)

lines.append(
    f"- strong-shift sensors vs test: "
    f"{len(high_shift)} / {len(shift_df)}"
)

lines.append(
    f"- test exact-overlap rows with supervised: "
    f"{test_overlap_rows} / {len(test)} "
    f"({test_overlap_rows / len(test):.3%})"
)

lines.append(
    f"- test rows matching supervised positive-only vectors: "
    f"{positive_overlap}"
)

lines.append(
    f"- test rows matching supervised conflict vectors: "
    f"{conflict_overlap}"
)

if len(
    coverage_df
):
    for _, r in coverage_df.iterrows():
        lines.append(
            f"- {r['dimension']} test category coverage: "
            f"{r['intersection']}/{r['test_unique']} "
            f"({r['test_category_coverage']:.1%})"
        )

for line in lines:
    print(
        line
    )

with open(
    OUT_DIR
    / "supervised_coverage_summary.txt",
    "w",
    encoding="utf-8",
) as f:

    f.write(
        "\n".join(
            lines
        )
    )


# =============================================================================
# 완료
# =============================================================================

banner(
    "완료"
)

print(
    "결과 폴더:",
    OUT_DIR,
)

print()

print(
    "핵심 파일:"
)

print(
    "  1) supervised_vs_test_shift.csv"
)

print(
    "  2) metadata_test_coverage.csv"
)

print(
    "  3) supervised_defect_by_date.csv"
)

print(
    "  4) test_overlap_by_supervised_target.csv"
)

print(
    "  5) moldset1211_vs_supervised6736.csv"
)

print(
    "  6) supervised_coverage_summary.txt"
)

print()

print(
    "실행 후 4) metadata coverage, "
    "5) defect coverage, "
    "6) metadata coverage vs Test, "
    "7) sensor shift, "
    "8~9) overlap, "
    "11) 자동 요약을 보내주세요."
)
